# Stage 3 — Gender Tagging of Cited Figures

**Goal:** For each unique person name in `citations_clean.csv`, query Wikidata to find their gender.

By the end of this notebook you will have:
- `data/gender_lookup.csv` — a table mapping each cited name → Wikidata gender (male/female/unknown)
- `data/citations_with_gender.csv` — the full citation dataset enriched with cited-person gender

**Why Wikidata?** It is a structured knowledge base covering all major French politicians from 1973–1993. We query it via SPARQL — a query language for linked databases.

## 0. Setup

In [1]:
import pandas as pd
import requests
import time
import json
from pathlib import Path
from tqdm import tqdm

PROJECT_ROOT = Path("..")
SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"
HEADERS = {"User-Agent": "NER-gender-project/1.0 (anahireyes08@gmail.com)"}

df_citations = pd.read_csv(PROJECT_ROOT / "data" / "citations_clean.csv")
print(f"Citations loaded: {len(df_citations):,}")
print(f"Unique cited names: {df_citations['cited_person_clean'].nunique():,}")

Citations loaded: 50,416
Unique cited names: 10,139


## 1. Query Strategy

We only query the **top 500 most-cited names** — they cover the vast majority of citations.
Names cited only 2–3 times are mostly OCR noise or very minor figures; labeling them has little impact on the analysis.

For each name we send a SPARQL query to Wikidata that:
1. Searches for a human entity (`wdt:P31 wd:Q5`) whose French or English label matches the name
2. Returns their `sex or gender` property (`wdt:P21`)

We add a 0.3s delay between requests to respect Wikidata's rate limit.

In [2]:
MW_API = "https://www.wikidata.org/w/api.php"

def query_wikidata_gender(name):
    """
    Two-step Wikidata lookup via MediaWiki API (faster than SPARQL):
    1. wbsearchentities — find Q-IDs matching the name
    2. wbgetentities    — fetch P31 (instance of human) and P21 (gender)
    Returns a dict or None if not found.
    """
    # Step 1: search by name
    try:
        resp = requests.get(
            MW_API,
            params={
                "action": "wbsearchentities",
                "search": name,
                "language": "fr",
                "type": "item",
                "limit": 5,
                "format": "json",
            },
            headers=HEADERS,
            timeout=15,
        )
        resp.raise_for_status()
        results = resp.json().get("search", [])
    except Exception as e:
        print(f"  Search error for '{name}': {e}")
        return None

    if not results:
        return None

    # Step 2: for each candidate, check it's a human and get gender
    qids = [r["id"] for r in results]
    try:
        resp2 = requests.get(
            MW_API,
            params={
                "action": "wbgetentities",
                "ids": "|".join(qids),
                "props": "claims|labels",
                "languages": "fr|en",
                "format": "json",
            },
            headers=HEADERS,
            timeout=15,
        )
        resp2.raise_for_status()
        entities = resp2.json().get("entities", {})
    except Exception as e:
        print(f"  Entity fetch error for '{name}': {e}")
        return None

    GENDER_MAP = {
        "Q6581097": "male",
        "Q6581072": "female",
        "Q1097630": "intersex",
        "Q2449503": "transgender female",
        "Q48270":   "non-binary",
    }

    for qid in qids:
        entity = entities.get(qid, {})
        claims = entity.get("claims", {})
        # Must be instance of human (P31 = Q5)
        p31 = [c["mainsnak"]["datavalue"]["value"]["id"]
               for c in claims.get("P31", [])
               if c["mainsnak"].get("datavalue")]
        if "Q5" not in p31:
            continue
        # Get gender (P21)
        p21 = [c["mainsnak"]["datavalue"]["value"]["id"]
               for c in claims.get("P21", [])
               if c["mainsnak"].get("datavalue")]
        gender = GENDER_MAP.get(p21[0], "other") if p21 else "unknown"
        label = (entity.get("labels", {}).get("fr", {}) or
                 entity.get("labels", {}).get("en", {})).get("value", "")
        return {"wikidata_id": qid, "wikidata_label": label, "wikidata_gender": gender}

    return None

# Quick test
result = query_wikidata_gender("François Mitterrand")
print(result)
result2 = query_wikidata_gender("Arlette Laguiller")
print(result2)

{'wikidata_id': 'Q2038', 'wikidata_label': 'François Mitterrand', 'wikidata_gender': 'male'}
{'wikidata_id': 'Q12942', 'wikidata_label': 'Arlette Laguiller', 'wikidata_gender': 'female'}


## 2. Run queries for top 1000 cited names

This takes ~3 minutes (500 names × 0.3s delay). Results are cached to a JSON file so you never need to re-query.

In [3]:
CACHE_PATH = PROJECT_ROOT / "data" / "wikidata_cache.json"

# Load existing cache if available
if CACHE_PATH.exists():
    with open(CACHE_PATH) as f:
        cache = json.load(f)
    print(f"Loaded cache with {len(cache)} entries")
else:
    cache = {}

# Get top 1000 names by citation frequency
top_names = (
    df_citations["cited_person_clean"]
    .value_counts()
    .head(1000)
    .index.tolist()
)
to_query = [n for n in top_names if n not in cache]
print(f"Names to query: {len(to_query)} (already cached: {len(top_names) - len(to_query)})")

for name in tqdm(to_query, desc="Querying Wikidata"):
    result = query_wikidata_gender(name)
    cache[name] = result
    time.sleep(0.3)

# Save updated cache
with open(CACHE_PATH, "w") as f:
    json.dump(cache, f, ensure_ascii=False, indent=2)
print(f"Cache saved ({len(cache)} entries total)")

found = sum(1 for v in cache.values() if v is not None)
print(f"Found on Wikidata: {found} / {len(cache)} ({found/len(cache)*100:.1f}%)")

Loaded cache with 1000 entries
Names to query: 206 (already cached: 794)


Querying Wikidata: 100%|██████████| 206/206 [03:00<00:00,  1.14it/s]

Cache saved (1206 entries total)
Found on Wikidata: 1002 / 1206 (83.1%)


## 3. Build gender lookup table & join to citations

In [4]:
# Build lookup dataframe from cache
rows = []
for name, result in cache.items():
    if result:
        rows.append({
            "cited_person_clean": name,
            "wikidata_id":        result["wikidata_id"],
            "wikidata_label":     result["wikidata_label"],
            "cited_gender":       result["wikidata_gender"],
        })
    else:
        rows.append({
            "cited_person_clean": name,
            "wikidata_id":        None,
            "wikidata_label":     None,
            "cited_gender":       "unknown",
        })

df_lookup = pd.DataFrame(rows)
print("Gender distribution in lookup table:")
print(df_lookup["cited_gender"].value_counts())

# Save lookup
df_lookup.to_csv(PROJECT_ROOT / "data" / "gender_lookup.csv", index=False)
print("\nSaved gender_lookup.csv")

Gender distribution in lookup table:
cited_gender
male       953
unknown    207
female      46
Name: count, dtype: int64

Saved gender_lookup.csv


In [5]:
# Join gender back to citations
df_enriched = df_citations.merge(
    df_lookup[["cited_person_clean", "cited_gender", "wikidata_id", "wikidata_label"]],
    on="cited_person_clean",
    how="left"
)
df_enriched["cited_gender"] = df_enriched["cited_gender"].fillna("unknown")

print(f"Enriched citations: {len(df_enriched):,}")
print(f"\nCited-person gender distribution:")
print(df_enriched["cited_gender"].value_counts())

print(f"\nCoverage: {(df_enriched['cited_gender'] != 'unknown').mean()*100:.1f}% of citations have a known gender")

df_enriched.to_csv(PROJECT_ROOT / "data" / "citations_with_gender.csv", index=False)
print("\nSaved citations_with_gender.csv")

Enriched citations: 50,416

Cited-person gender distribution:
cited_gender
unknown    30651
male       18500
female      1265
Name: count, dtype: int64

Coverage: 39.2% of citations have a known gender

Saved citations_with_gender.csv


## 4. Sanity check — who are the female figures?

Before moving to analysis, verify: are the women found by Wikidata real female political figures?

In [7]:
female_figures = df_enriched[df_enriched["cited_gender"] == "female"]
print("=== Top 20 female figures cited overall ===")
print(female_figures["cited_person_clean"].value_counts().head(20).to_string())

print("\n=== Female figures cited by FEMALE candidates ===")
print(
    female_figures[female_figures["candidate_sex"] == "femme"]["cited_person_clean"]
    .value_counts().head(15).to_string()
)

print("\n=== Female figures cited by MALE candidates ===")
print(
    female_figures[female_figures["candidate_sex"] == "homme"]["cited_person_clean"]
    .value_counts().head(15).to_string()
)

=== Top 20 female figures cited overall ===
cited_person_clean
Arlette Laguiller        674
Huguette Bouchardeau      84
Gisele Halimi             84
Simone De Beauvoir        41
Dominique Voynet          27
Simone Veil               19
Gisele Moreau             17
Adrienne Horvath          14
Yvette Roudy              14
Edith Cresson             14
Catherine Lalumiere       14
Helene Constans           13
Rolande Perlican          12
Jacqueline Chonavel       12
Marie-France Stirbois     12
Nicole Pery               11
Helene Missoffe           11
Louise Moreau             11
Veronique Neiertz         11
Marie Jacq                10

=== Female figures cited by FEMALE candidates ===
cited_person_clean
Arlette Laguiller       268
Gisele Halimi            81
Simone De Beauvoir       40
Huguette Bouchardeau     20
Gisele Moreau            15
Catherine Lalumiere      13
Rolande Perlican         12
Louise Moreau            11
Adrienne Horvath         11
Yvette Roudy             11
Veroniq

## 5. Summary

- `data/gender_lookup.csv` — Wikidata gender for top 500 cited names
- `data/citations_with_gender.csv` — full enriched citation dataset
- `data/wikidata_cache.json` — raw cache (don't re-query what's already here)

**Next:** `04_analysis.ipynb` — test whether female candidates cite proportionally more women than male candidates do.